## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Bugs per thousand lines by release: u chart

**u Control Chart** · a worked example from the StatNex teaching guide

**The story.** A software team counts the bugs found in each of 25 releases. Releases differ in size (16 to 41 thousand lines of changed code), so the team compares bugs per thousand lines. Release 13 was rushed out before a deadline.

**The question.** Is any release buggier than the team's normal standard, once release size is taken into account?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import IPython.display
import matplotlib

matplotlib.use("Agg")


def show_new_figures(*_, shown=set()):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path.name, path.stat().st_mtime_ns) not in shown:
            shown.add((path.name, path.stat().st_mtime_ns))
            IPython.display.display(IPython.display.Image(filename=str(path)))


events = IPython.get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `10_control_chart/u_chart.csv` (25 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Release,Code changed (thousand lines),Bugs found
1.0,34.0,25.0
2.0,17.0,9.0
3.0,37.0,13.0
4.0,16.0,7.0
5.0,18.0,6.0
6.0,19.0,8.0
7.0,24.0,17.0
8.0,23.0,15.0
9.0,26.0,13.0
10.0,24.0,16.0
11.0,34.0,17.0
12.0,29.0,16.0
13.0,36.0,46.0
14.0,22.0,9.0
15.0,31.0,22.0
16.0,33.0,18.0
17.0,38.0,13.0
18.0,41.0,20.0
19.0,37.0,16.0
20.0,19.0,8.0
21.0,20.0,8.0
22.0,36.0,17.0
23.0,39.0,19.0
24.0,38.0,16.0
25.0,16.0,8.0
""")

## Helper: control_signals

A step that no library does, written out. In the code download it is the file `control_signals.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Reading a Shewhart control chart: the two pattern rules, and the chart itself.

SciPy and statsmodels have no control charts, so both are written out here.

A point outside the 3-sigma limits is the obvious signal. A chart is also read for its
shape, because two patterns inside the limits are about as unlikely as a point outside
when nothing has changed (Nelson's rules 2 and 3; ISO 7870-2 and JIS Z 9020-2 list the
same tests):

* 9 points in a row on one side of the centre line -- the level may have shifted;
* 6 rises in a row (7 points climbing), or 6 falls -- a trend.

These are the two StatNex applies. Each run is reported once, from its first point to
its last, however long it lasts: nine in a row sets the warning off, and a run of
eighteen is still one run. A point exactly on the centre line, or a missing one (the
first moving range), breaks a run.
"""

import numpy as np
from matplotlib.figure import Figure


def stretches(signs, minimum):
    """Each stretch of at least `minimum` equal, non-zero signs, as (first, last) positions."""
    found, start = [], 0
    for i in range(1, len(signs) + 1):
        if i == len(signs) or signs[i] != signs[start]:
            if signs[start] != 0 and i - start >= minimum:
                found.append((start, i - 1))
            start = i
    return found


def pattern_signals(label, values, centre):
    """Messages for the runs and trends in one chart's points (a pandas Series)."""
    numbers = list(values.index)
    points = values.to_numpy(float)
    found = []

    side = np.where(np.isfinite(points), np.sign(points - np.asarray(centre, float)), 0.0)
    for first, last in stretches(side, 9):
        where = "above" if side[first] > 0 else "below"
        found.append(f"{label}: points {numbers[first]} to {numbers[last]} are {last - first + 1} in a row "
                     f"{where} the centre line. The level of the process may have shifted.")

    # Step i joins points i - 1 and i; six steps one way are seven points climbing or falling.
    steps = np.zeros(len(points))
    for i in range(1, len(points)):
        if np.isfinite(points[i]) and np.isfinite(points[i - 1]):
            steps[i] = np.sign(points[i] - points[i - 1])
    for first, last in stretches(steps, 6):
        where = "rise" if steps[first] > 0 else "fall"
        found.append(f"{label}: points {numbers[first - 1]} to {numbers[last]} {where} "
                     f"monotonically. This is a sign of a change in trend.")
    return found


def draw(path, title, values, centre, upper, lower, xlabel="Subgroup"):
    """One chart: the points, the centre line, and the limits (stepped if they vary)."""
    fig = Figure(figsize=(7, 4.5))
    ax = fig.subplots()
    x = np.asarray(values.index)
    ax.plot(x, values.to_numpy(float), marker="o", color="black", linewidth=1)
    for line, style in ((upper, "--"), (centre, "-"), (lower, "--")):
        ax.step(x, np.broadcast_to(np.asarray(line, float), x.shape), where="mid",
                linestyle=style, color="grey", linewidth=1)
    out = (values > upper) | (values < lower)
    ax.plot(x[out.to_numpy()], values[out].to_numpy(float), "o", color="red")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    fig.savefig(path, dpi=150)

## Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
# draw and pattern_signals: defined in the helper cell control_signals above (control_signals.py in the code download)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# One row per subgroup: how many units were inspected (an area, a length, a count of
# items, thousands of lines of code) and how many nonconformities were found. Blanks,
# text, negative counts or a unit count of 0 stop the chart.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
units = pd.to_numeric(data["Code changed (thousand lines)"], errors="coerce")
defects = pd.to_numeric(data["Bugs found"], errors="coerce")
if units.isna().any() or defects.isna().any() or (units <= 0).any() or (defects < 0).any():
    raise ValueError("Every subgroup needs a unit count above 0 and a count of 0 or more.")
units.index = defects.index = np.arange(1, len(units) + 1)  # subgroup numbers 1, 2, ...


def chart(keep):
    """Poisson limits that move with each subgroup's size: more units pin u down."""
    # Each count is treated as Poisson (defects turning up independently at a steady rate
    # per unit). The rate is estimated by u-bar, all nonconformities over all units, and
    # each subgroup gets its own limits (n_i = units inspected):
    #   CL = u-bar,  UCL_i, LCL_i = u-bar +/- 3 sqrt(u-bar / n_i)
    u_bar = defects[keep].sum() / units[keep].sum()
    margin = 3.0 * np.sqrt(u_bar / units[keep])
    return pd.DataFrame({"u": defects[keep] / units[keep], "CL": u_bar,
                         "UCL": u_bar + margin, "LCL": np.maximum(u_bar - margin, 0.0)})


def outside(table):
    return (table["u"] > table["UCL"]) | (table["u"] < table["LCL"])


keep = pd.Series(True, index=units.index)

## Control chart data

In [ ]:
# One row per subgroup: its rate per unit (u) against the centre line (CL) and that
# subgroup's own limits (UCL, LCL), wide for a small subgroup and narrow for a big one.
# A rate outside its limits is a signal that something other than everyday variation
# affected that subgroup. A lower limit that would fall below 0 is drawn at 0.
table = chart(keep)
print()
print(table)
print(f"\nSubgroups outside the limits: {list(table.index[outside(table)])}")

## The chart

In [ ]:
# Besides points outside the limits, a chart is read for patterns that are just as
# unlikely by chance: 9 points in a row on one side of the centre line, or 6 rises (or
# falls) in a row. A long run below the centre line
# is a signal too: the rate may have improved.
for message in pattern_signals("u chart", table["u"], table["CL"]):
    print(message)
Path("figures").mkdir(exist_ok=True)
draw("figures/u_chart.png", "u chart", table["u"], table["CL"], table["UCL"], table["LCL"])

## What the result says

A u chart follows a rate, here bugs found per thousand lines of changed code, so that large and small releases can be compared. The centre line (CL) is the usual rate: 0.54 bugs per thousand lines. Each release has its own control limits, the reach of ordinary chance variation, because a larger release gives a more exact rate.

Only release 13, the rushed one, is outside its limits. Its rate is 1.28, more than twice the usual rate, and its upper control limit (UCL) is 0.91.

The last nine releases, 17 to 25, all sit below the centre line, so the rate of bugs found may have fallen lately. The chart cannot tell whether that means better code or less testing.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.